In [1]:
from sklearn.base import BaseEstimator, TransformerMixin
from transformers import pipeline
import torch

class HuggingFaceSummarizer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name="sshleifer/distilbart-cnn-12-6", max_length=40, min_length=10):
        self.model_name = model_name
        self.max_length = max_length
        self.min_length = min_length
        self.summarizer = None
        self.device = 0 if torch.cuda.is_available() else -1

    def fit(self, X, y=None):
        if self.summarizer is None:
            self.summarizer = pipeline("summarization", model=self.model_name, device=self.device)
        return self

    def transform(self, X):
        if self.summarizer is None:
            self.summarizer = pipeline("summarization", model=self.model_name, device=self.device)
        
        result = self.summarizer(
            X,
            max_length = self.max_length,
            min_length = self.min_length,
            truncation = True
        )
        return [res['summary_text'] for res in result]

c:\Homework\Code File\Python Code File\Pratice\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Small test

In [2]:
X_long_texts = [
    "I've been using this vacuum cleaner for about three weeks now. At first, I struggled with the attachments, and the manual wasn't very clear. However, once I figured out how the motorized brush works, it easily picked up all the pet hair on my rugs. Overall, it's a solid machine, though a bit heavy to carry up the stairs.",
    "The delivery was delayed by four days, which was incredibly frustrating because I needed it for a weekend trip. When the backpack finally arrived, the zipper snagged immediately. I tried to fix it, but the fabric feels cheap and flimsy. I will definitely be returning this and asking for a full refund.",
]
 
y_labels = ["positive", "negative"]

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

classification_pipeline = Pipeline([
    ('summarizer', HuggingFaceSummarizer(max_length=30, min_length=10)),
    ('vectorizer', TfidfVectorizer()), # Used to encode build numerical text representations, needed for ML
    ('classifier', LogisticRegression())
])

In [4]:
classification_pipeline.fit(X_long_texts, y_labels)
 
print("Pipeline trained successfully on summarized reviews!")

c:\Homework\Code File\Python Code File\Pratice\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Pipeline trained successfully on summarized reviews!


In [5]:
# In kết quả tóm tắt ra màn hình
summarizer = classification_pipeline.named_steps["summarizer"]
summaries = summarizer.transform(X_long_texts)

for i, (original, summary) in enumerate(zip(X_long_texts, summaries), start=1):
    print(f"--- Đánh giá {i} ---")
    print(f"Tóm tắt: {summary}\n")
    print(f"Gốc ({len(original)} ký tự): {original}\n")

--- Đánh giá 1 ---
Tóm tắt:  Overall, it's a solid machine, though a bit heavy to carry up the stairs . At first, I struggled with the attachments,

Gốc (322 ký tự): I've been using this vacuum cleaner for about three weeks now. At first, I struggled with the attachments, and the manual wasn't very clear. However, once I figured out how the motorized brush works, it easily picked up all the pet hair on my rugs. Overall, it's a solid machine, though a bit heavy to carry up the stairs.

--- Đánh giá 2 ---
Tóm tắt:  The delivery was delayed by four days, which was incredibly frustrating . The zipper snagged immediately . The fabric feels cheap and flimsy .

Gốc (302 ký tự): The delivery was delayed by four days, which was incredibly frustrating because I needed it for a weekend trip. When the backpack finally arrived, the zipper snagged immediately. I tried to fix it, but the fabric feels cheap and flimsy. I will definitely be returning this and asking for a full refund.

